<div style="text-align: center">
<img src="https://github.com/LinkedEarth/Logos/blob/master/PyleoTUPS/pyleotups_logo.png?raw=true" alt="PyleoTUPS logo" width="400">
</div>

# Working with NOAA NCEI for Paleoclimatology

## Authors

Deborah Khider [![ORCID](https://img.shields.io/badge/ORCID-0000--0001--7501--8430-A6CE39?logo=orcid)](https://orcid.org/0000-0001-7501-8430), Dhiren Oswal [![ORCID](https://img.shields.io/badge/ORCID-0009--0001--2495--2626-A6CE39?logo=orcid)](https://orcid.org/0009-0001-2495-2626)


## Preamble

The goal of this tutorial is to familiarize you with the `NOAADataset` object and its functionalities in PyleoTUPS. In this tutorial, we focus specifically on searching by NOAA study ID; a separate tutorial will cover the full range of NOAA search capabilities.

In the NOAA framework, study IDs represent a *publication unit* for a dataset—that is, all data associated with a given study. For example, a study may include multiple data tables corresponding to different sites. While these tables share the same study ID, each site is assigned a unique site ID.


### Goals

 - Search for datasets using a specific study ID
 - Obtain information such as location, publication, variable metadata and the associated data.

### Pre-requisites

- Understanding of NOAA datasets

### Reading time

15 min

Let's import our packages!

In [1]:
import pyleotups as pt
import pandas as pd

## Accessing a dataset using the NOAA study ID

This is the most basic search you can perform, but if you know which dataset you are looking for, it is the easiest way to get all the needed information. First, you need to create a [`NOAADataset` object](https://pyleotups.readthedocs.io/en/latest/api.html#noaadataset-pyleotups-core-noaadataset-noaadataset) that will store the information.

In [2]:
ds=pt.NOAADataset()

Let's do a simple search, knowing the NOAA study ID. For this example, let's use the dataset from [Clemens et al. (2021)](https://doi.org/10.1126/sciadv.abg3848), which can be accessed through [NOAA Paleo portal](https://www.ncei.noaa.gov/access/paleo-search/study/33213). The study contains several tables of measurements made on a marine sediment at site IODP U1446:

<div style="
    padding: 10px; 
    background-color: #fff3cd; 
    border-left: 6px solid #ffcc00; 
    margin: 10px 0;">
    <strong>Warning:</strong> Even for one study, the search may take some time. 
</div>

In [3]:
res = ds.search_studies(noaa_id=33213)

[2026-05-15 15:03:29,928][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&NOAAStudyId=33213&limit=100


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 2376.38it/s]
[2026-05-15 15:03:30,575][INFO] - Retrieved 1 studies.


<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> The NOAADataset object will only accept one ID at a time. You cannot pass a list of various IDs to get multiple NOAA studies at a time.
</div>

The [`get_summary` method](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_summary) provides basic information about the dataset, such as the name of the study, the NOAA [DataType](https://www.ncei.noaa.gov/products/paleoclimatology), the time coverage and the associated publication. The function returns a `pandas.DataFrame`.

In [4]:
df_summary = ds.get_summary()
display(df_summary)

,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,33213,74834,"Bay of Bengal, Northeast Indian Margin Stable ...",PALEOCEANOGRAPHY,1462580,280,-1460630,1670,"(19.083, 19.083, 85.733, 85.733)","Provided Keywords: Indian monsoon, South Asian...",None,"Steven Clemens, Masanobu Yamamoto, Kaustubh Th...","[{'Author': 'Clemens, Steven; Yamamoto, Masano...","[[{'DataTableID': '45857', 'DataTableName': 'U...",[{'fundingAgency': 'US National Science Founda...


Let's have a look at the information returned in this DataFrame:

* `StudyID`: Matches the value we searched for. While redundant in this case, it becomes useful when querying using other criteria.
* `XMLID`: A different identifier for the same study. Each `StudyID` is associated with a corresponding `XMLID`.
* `Study Name`: The name of the study.
* `DataType`: Uses NOAA’s nomenclature for the data type.
* `EarliestYearBP`, `MostRecentYearBP`, `EarliestYearCE`, `MostRecentYearCE`: The temporal bounds of the dataset.
* `Coverage`: The geographic area covered by the study. In this example, it corresponds to a single point. This information can also be retrieved using the `get_geo` function.
* `StudyNotes`: Includes descriptive notes and keywords associated with the study.
* `ScienceKeywords`: Standardized science keywords.
* `Investigators`: The contributors to the study.
* `Publications`: Information about publications associated with the dataset. This can also be retrieved using the `get_publication` function.
* `Sites`: The sites associated with the NOAA study. A single study may include multiple sites, and each site may contain multiple data tables. This information can be retrieved using the `get_sites` function.
* `Funding`: Information about funding sources for the study. This can be retrieved using the `get_funding` function.

This function provides a high-level overview of the study and corresponds to the information available on the [NOAA landing page](https://www.ncei.noaa.gov/access/paleo-search/study/33213).

To facilitate downstream analysis, we provide additional functions that extract key components of the study into separate `Pandas.DataFrame` objects, making it easier to work with specific aspects of the data.

### Obtaining Information about Funding and Publications

To get more details about the [funding](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_funding):

In [5]:
df_funding = ds.get_funding()
display(df_funding)

,StudyID,StudyName,FundingAgency,FundingGrant
0,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",US National Science Foundation,OCE1634774
1,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",United States Geological Survey (USGS),NE/L002493/1
2,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",UK Natural Environment Research Council (NERC),"JPMXS05R2900001, 19H05595"
3,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",Japan Society for the Promotion of Science (JSPS),JPMXS05R2900001


Note that each table has a studyID key. You can think of this as the key in relational databases. It may not matter right now, but this will become useful as you use more advanced search functionalities which will return several studies.

In addition to funding information, `PyleoTUPS` allows you to get information about the [publication(s)](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_publications) associated with the dataset. 

In [6]:
bib, df_pub = ds.get_publications()
display(df_pub)

,Author,Title,Journal,Year,Volume,Number,Pages,Type,DOI,URL,CitationKey,StudyID,StudyName
0,"Clemens, Steven; Yamamoto, Masanobu; Thirumala...",Remote and Local Drivers of Pleistocene South ...,Science Advances,2021,7,23,NaN,publication,10.1126/sciadv.abg3848,http://dx.doi.org/10.1126/sciadv.abg3848,M._Remote_2021_33213,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."


<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> You can set the `save` parameter to True if you want to save a copy of the BibTeX entries.
</div>

### Obtaining Information about Geographical Location

Similarly, you can return information about the [location](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_geo) of the record: 

In [7]:
df_geo = ds.get_geo()
display(df_geo)

,StudyID,DataType,SiteID,SiteName,LocationName,GeoType,GeometryType,MinLatitude,MaxLatitude,MinLongitude,MaxLongitude,MinElevation,MaxElevation
0,33213,PALEOCEANOGRAPHY,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440


When getting the geographical coordinates for a study, each row in the returned DataFrame will correspond to a specific site. In this case, there is only one site in the study. We will be looking at a more interesting case later in this tutorial. 

### Obtaining Variable Information

The next step is to actually obtain the data tables present in the dataset. Doing so requires some understanding of how NOAA organizes their database. 

At the top level, there is the study, which is represented by a unique studyID and xmlID. Each study can have multiple sites. For instance, a study could be a compilation of benthic $\delta^{18}O$ records, each with their own site. Each site can have multiple tables, which might refer to different measurements (e.g., one table for Mg/Ca and SST and another one for planktic $\delta^{18}O$). PyleoTUPS keeps this structure intact to give you as much flexibility in your workflows as possible. 

Let's first have a look at the [sites](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_sites):

In [8]:
df_sites = ds.get_sites()
display(df_sites)

,SiteID,SiteName,StudyID,FileURL,Variables,TotalFilesAvailable,LocationName,GeoType,GeometryType,MinLatitude,MaxLatitude,MinLongitude,MaxLongitude,MinElevation,MaxElevation,StudyName
0,58697,IODP U1446,33213,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[Site, d13C, d18O, Age, Sample_Depth, Section...",8,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,"Bay of Bengal, Northeast Indian Margin Stable ..."


As you may notice, this function returns similar information than the `get_geo` method. This allows multiple entry points to the database and functionality harmonization between PANGAEA and NOAA NCEI. Which one should you use? Whichever is the most convenient for you and your workflow. For instance, if interested in a meta-analysis of available records on NOAA NCEI and PANGAEA, `get_geo` might be more convenient as the two APIs have been harmonized between the two databases. But if familiar with the NOAA structure and only working with datasets stored there, the `get_sites` might be a more useful and natural function to use. 

Whichever you choose, you do not need to first get the sites to access the [data tables](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_tables), as shown below: 

In [9]:
df_tables = ds.get_tables()
display(df_tables)

,DataTableID,DataTableName,TimeUnit,FileURL,Variables,FileDescription,SiteID,SiteName,LocationName,GeoType,GeometryType,MinLatitude,MaxLatitude,MinLongitude,MaxLongitude,MinElevation,MaxElevation,StudyID,StudyName
0,45857,U1446 Benthic Isotopes Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, d13C, d18O, Age, Sample_Depth, Section_...",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
1,45864,U1446 Age Model Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Age, Sample_Depth, Comments, Site]",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
2,45863,U1446 Rb/Ca Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Rb/Ca, Rb_normalized, Ca_normalized, Age, Sam...",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
3,45862,U1446 Mg/Ca Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Analytical_Facility, Type, Hole, Site, Mg/Ca_...",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
4,45861,U1446 LeafWax CarbonIsotope Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[d13C_Ave, d13C_C32, d13C_C30, d13C_C28, d13C_...",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
5,45860,U1446 d18Osw Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[d18Osw_out_Mg/Ca, Mg/Ca, d18Osw_out_Shakun, S...",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
6,45859,U1446 TEX86H_SST Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[SST, TEX86H, Age, Sample_Depth, Section_Depth...",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
7,45858,U1446 Planktic Isotopes Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[d13C, d18O, Age, Sample_Depth, Section_Depth,...",NOAA Template File,58697,IODP U1446,Ocean>Indian Ocean,Feature,POINT,19.083,19.083,85.733,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."


This function operates directly on the NOAA object and does not require a siteID.

Note that each `TableID` is unique and can be used to get more information about the data. Most of the fields returned here are self-explanatory, but let's go over the few that require further explanation and design consideration:
* `fileURL`: This is where the actual data lives! So far, all the information we have gathered has been made available through the NOAA API response in a JSON file. Now it is time to actually read the data!
* `FileDescription`: This is an internal representation for NOAA and does inform about the type of files to be found at the `fileURL`. PyleoTUPS can only read text files, which means file description such as NOAA Template File, NOAA Template File - Sunthesis Metadata, Raw Measurements - NOAA Template File, Chronology - NOAA Template File. However, PyleoTUPS will not work on other format such as NetCDF (we recommend the [`xarray` library](https://xarray.dev) for these files), `.lpd` (we have created a [library to deal with LiPD files called `PyLiPD`](https://pylipd.readthedocs.io/en/latest/)), html, or pickle file.

<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> Let's make matters more complicated! The same TableID can be used multiple times for different file format. For instance, if the data uses the NOAA template and LiPD file format. AND a TableID (which is really the file) can actually have multiple tables in them.
</div>

According to the table above, our unique study contains one unique site (SiteID = 58697) and eight tables. To get the data, you need to pass the `DataTableID` or `FileURL` to the [`get_data()` method](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_data). Let's have a look at the TEX86 data:

In [10]:
dfs = ds.get_data(dataTableIDs="45859")
type(dfs)

list

Notice that we are returning a list. This is because in some cases, the PyleoTUPS parser may identify more than one table in the file. This is most common with older studies. 

Let's have a look at the first and (only) table:

In [11]:
display(dfs[0])

,Site,Hole,Core,Type,Section,Section_Depth,Sample_Depth,Age,TEX86H,SST
0,U1446,C,1,H,1,4.5,0.045,0.31,-0.1177,30.55
1,U1446,C,1,H,1,31.5,0.315,0.66,-0.1216,30.28
2,U1446,C,1,H,1,61.5,0.615,1.04,-0.1183,30.51
3,U1446,C,1,H,1,90.5,0.905,1.41,-0.1089,31.15
4,U1446,C,1,H,1,121.5,1.215,1.80,-0.1155,30.70
...,...,...,...,...,...,...,...,...,...,...
605,U1446,C,23,F,4,7,203.365,1458.08,-0.1535,28.10
606,U1446,C,23,F,4,37,203.665,1459.49,-0.1640,27.38
607,U1446,C,23,F,4,67,203.965,1460.90,-0.1592,27.71
608,U1446,C,23,F,4,97,204.265,1462.32,-0.1649,27.32


And we have the data! How about more metadata about the variables themselves?

Well, there is a [function](https://pyleotups.readthedocs.io/en/latest/api.html#pyleotups.core.NOAADataset.NOAADataset.get_variables) for this as well:

In [12]:
df_var = ds.get_variables(dataTableIDs="45859")
display(df_var)

,StudyID,SiteID,FileURL,VariableName,cvDataType,cvWhat,cvMaterial,cvError,cvUnit,cvSeasonality,cvDetail,cvMethod,cvAdditionalInfo,cvFormat,cvShortName
DataTableID,,,,,,,,,,,,,,,
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,SST,CLIMATE RECONSTRUCTIONS|PALEOCEANOGRAPHY,earth system variable>temperature variable>tem...,reconstruction material>organic compound index...,None,temperature unit>degree Celsius,None,None,NaN,index of Kim et al. 2010,Numeric,SST
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,TEX86H,PALEOCEANOGRAPHY,chemical composition>compound>organic compound...,NaN,None,dimensionless,None,None,laboratory method>chromatography>liquid chroma...,index of Schouten et al. 2002; extracted lipids,Numeric,TEX86H
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Age,CLIMATE RECONSTRUCTIONS|PALEOCEANOGRAPHY,age variable>age,NaN,None,time unit>age unit>calendar kiloyear before pr...,None,None,NaN,NaN,Numeric,Age
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Sample_Depth,CLIMATE RECONSTRUCTIONS|PALEOCEANOGRAPHY,depth variable>depth,NaN,None,length unit>meter,None,None,NaN,Spliced core composite depth below sea floor ...,Numeric,Sample_Depth
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Section_Depth,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,length unit>centimeter,None,None,NaN,Mid-depth of sample within section,Numeric,Section_Depth
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Core,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,Core number,Numeric,Core
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Section,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,Section number ( 1 through 7 and core catcher ...,Character,Section
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Type,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,H (9 m hydraulic piston core) F (4.5 m hydrau...,Character,Type
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Hole,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,Hole drilled at Site U1446,Character,Hole


For more information about the meaning of the columns and possible values, please have a look at the [NOAA PaST Thesaurus](https://www.ncei.noaa.gov/products/paleoclimatology/paleoenvironmental-standard-terms-thesaurus). 

<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> You can pass multiple tables IDs as a list. The function will always return a list of DataFrames (hence why we selected the first one in the code above.)
</div>

Some relevant metadata for each column is also stored in the DataFrame attributes:

In [13]:
dfs[0].attrs

{'variables': ['Site',
  'Hole',
  'Core',
  'Type',
  'Section',
  'Section_Depth',
  'Sample_Depth',
  'Age',
  'TEX86H',
  'SST'],
 'NOAAStudyId': '33213',
 'StudyName': 'Bay of Bengal, Northeast Indian Margin Stable Isotope, Biomarker and SST Reconstructions since the Mid-Pleistocene'}

Instead of the table ID, you can also pass the file URL to get the data:

In [14]:
file_url = df_tables['FileURL'].iloc[0]
file_url

'https://www.ncei.noaa.gov/pub/data/paleo/contributions_by_author/clemens2021/clemens2021-u1446-benth-iso-noaa.txt'

In [15]:
df1 = ds.get_data(file_urls=file_url)[0]
display(df1.head())

,Site,Hole,Core,Type,Section,Section_Depth,Sample_Depth,Age,d18O,d13C
0,U1446,C,1,H,1,5,0.05,0.32,2.33,0.30
1,U1446,C,1,H,1,32,0.32,0.66,2.40,0.37
2,U1446,C,1,H,1,62,0.62,1.05,2.34,0.34
3,U1446,C,1,H,1,91,0.91,1.42,2.35,0.31
4,U1446,C,1,H,1,122,1.22,1.81,2.34,0.31


You can choose the method that is more convenient for you. 



### Summary

The functionalities created in PyleoTUPS allow you to access all the metadata store at NOAA NCEI for Paleo using various functions that mimics the structure offered by this data provider. Note that some information (e.g., geography) can be obtained through multiple functions. This built-in redundancy allows you to work with the methods that work best for you and your use case. 


## Dataset with Multiple Sites

As mentioned, some studies may have multiple sites. Let's see how this affect the functionalities. For this example, let's have a look a the paleoceanographic study for the Makassar Strait (Indo-Pacific Warm Pool) by [Linsley et al. (2010)](htpps://doi.org/10.1038/ngeo920), which can be found [here](https://www.ncei.noaa.gov/access/paleo-search/study/10420).

In [16]:
ds2 = pt.NOAADataset()

In [17]:
res = ds2.search_studies(noaa_id = 10420)
display(res)

[2026-05-15 15:04:06,053][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&NOAAStudyId=10420&limit=100


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 1766.77it/s]
[2026-05-15 15:04:06,530][INFO] - Retrieved 1 studies.


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,10420,9073,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...,PALEOCEANOGRAPHY,25900,0,-23950,1950,"(-13.083, 8.783, 115.2, 133.433)",Planktonic foraminiferal Mg/Ca records of mixe...,"[Other Hydroclimate Reconstruction, Sea Surfac...","Braddock Linsley, Yair Rosenthal, Delia Oppo","[{'Author': 'Linsley, B.K., Y. Rosenthal, and ...","[[{'DataTableID': '19200', 'DataTableName': '1...",[{'fundingAgency': 'US National Science Founda...


Let's first have a look at the geographical information: 

In [18]:
display(ds2.get_geo())

,StudyID,DataType,SiteID,SiteName,LocationName,GeoType,GeometryType,MinLatitude,MaxLatitude,MinLongitude,MaxLongitude,MinElevation,MaxElevation
0,10420,PALEOCEANOGRAPHY,37014,BJ8-03-10GGC,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.3667,-7.3667,115.250,115.250,-649,-649
1,10420,PALEOCEANOGRAPHY,37015,BJ8-03-13GGC,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.4000,-7.4000,115.200,115.200,-594,-594
2,10420,PALEOCEANOGRAPHY,54100,BJ8-03-70GGC,Ocean>Indian Ocean>Indonesia,Feature,POINT,-3.5660,-3.5660,119.383,119.383,-482,-482
3,10420,PALEOCEANOGRAPHY,60179,"Cores 13GGC, 70GGC, MD98-2162, MD98-2165",Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-9.6500,-3.5660,115.200,119.383,-649,-482
4,10420,PALEOCEANOGRAPHY,60180,"Cores MD97-2141, MD98-2181, MD98-2176",Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-5.0000,8.7830,121.283,133.433,-649,-482
5,10420,PALEOCEANOGRAPHY,60181,"Cores 13GGC, 70GGC, MD62, MD65, MD41, MD81, MD76",Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-9.6500,8.7830,115.200,133.433,-649,-482
6,10420,PALEOCEANOGRAPHY,60182,"Cores 13GGC, 70GGC, MD62, MD65, MD41, MD81, MD...",Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-13.0830,8.7830,115.200,133.433,-649,-482


There are a few takeaways here:
* As expected, there are multiple sites in this study. Each site is represented as a row
* The `GeometryType` indicates whether the site is one location (i.e., one physical archive), which is represented as a point or whether it represents multiple physical archives pulled together, which is represented as a polygon. 

Let's have a look at the site:

In [19]:
display(ds2.get_sites())

,SiteID,SiteName,StudyID,FileURL,Variables,TotalFilesAvailable,LocationName,GeoType,GeometryType,MinLatitude,MaxLatitude,MinLongitude,MaxLongitude,MinElevation,MaxElevation,StudyName
0,37014,BJ8-03-10GGC,10420,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[depth_top_cm, depth_bot_cm, age_top_calBP, d...",2,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.3667,-7.3667,115.250,115.250,-649,-649,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
1,37015,BJ8-03-13GGC,10420,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[age_calBP, depth_bot_cm, d18O_g.ruber, depth...",3,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.4000,-7.4000,115.200,115.200,-594,-594,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
2,54100,BJ8-03-70GGC,10420,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[age_calBP, d18O_g.ruber, SST, Mg/Ca_g.ruber,...",2,Ocean>Indian Ocean>Indonesia,Feature,POINT,-3.5660,-3.5660,119.383,119.383,-482,-482,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
3,60179,"Cores 13GGC, 70GGC, MD98-2162, MD98-2165",10420,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[SSTanom-SE, num_meas, compSST_anom, SST_SE, ...",2,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-9.6500,-3.5660,115.200,119.383,-649,-482,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
4,60180,"Cores MD97-2141, MD98-2181, MD98-2176",10420,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[age_calBP, num_meas, d18Osw_err, d18Osw_up, ...",1,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-5.0000,8.7830,121.283,133.433,-649,-482,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
5,60181,"Cores 13GGC, 70GGC, MD62, MD65, MD41, MD81, MD76",10420,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[d18Osw_err, d18Osw_up, d18Osw_lo, age_calBP,...",1,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-9.6500,8.7830,115.200,133.433,-649,-482,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
6,60182,"Cores 13GGC, 70GGC, MD62, MD65, MD41, MD81, MD...",10420,[https://www.ncei.noaa.gov/pub/data/paleo/cont...,"[[age_calBP, compSST_anom, num_meas, SST_SE, S...",2,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-13.0830,8.7830,115.200,133.433,-649,-482,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...


In this table, notice the various sites, with different coordinates and file URLs. This is an example of the one-to-many relationship that exists at NOAA: One study can have multiple sites and as we see in the next cells, one Site can have multiple Tables:

In [20]:
display(ds2.get_tables())

,DataTableID,DataTableName,TimeUnit,FileURL,Variables,FileDescription,SiteID,SiteName,LocationName,GeoType,GeometryType,MinLatitude,MaxLatitude,MinLongitude,MaxLongitude,MinElevation,MaxElevation,StudyID,StudyName
0,19200,10GGC_d18O,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[depth_top_cm, depth_bot_cm, age_top_calBP, d1...",NOAA Template File,37014,BJ8-03-10GGC,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.3667,-7.3667,115.250,115.250,-649,-649,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
1,51906,10GGC_d18O_bin,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[age_calBP, d18O_g.ruber]",NOAA Template File,37014,BJ8-03-10GGC,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.3667,-7.3667,115.250,115.250,-649,-649,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
2,19201,13GGC_d18O,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[age_calBP, depth_bot_cm, d18O_g.ruber, depth_...",NOAA Template File,37015,BJ8-03-13GGC,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.4000,-7.4000,115.200,115.200,-594,-594,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
3,51907,13GGC_bin,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[d18O_g.ruber, age_calBP, SST, d18Osw, Mg/Ca_g...",NOAA Template File,37015,BJ8-03-13GGC,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.4000,-7.4000,115.200,115.200,-594,-594,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
4,51908,13GGC_MgCaSST,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[depth_bot_cm, depth_top_cm, age_calBP, SST, M...",NOAA Template File,37015,BJ8-03-13GGC,Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POINT,-7.4000,-7.4000,115.200,115.200,-594,-594,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
5,19202,70GGC,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[age_calBP, d18O_g.ruber, SST, Mg/Ca_g.ruber, ...",NOAA Template File,54100,BJ8-03-70GGC,Ocean>Indian Ocean>Indonesia,Feature,POINT,-3.5660,-3.5660,119.383,119.383,-482,-482,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
6,51909,70GGC_bin,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[d18Osw, SST, Mg/Ca_g.ruber, d18O_g.ruber, age...",NOAA Template File,54100,BJ8-03-70GGC,Ocean>Indian Ocean>Indonesia,Feature,POINT,-3.5660,-3.5660,119.383,119.383,-482,-482,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
7,51910,SMAK_sst_comp,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[SSTanom-SE, num_meas, compSST_anom, SST_SE, S...",NOAA Template File,60179,"Cores 13GGC, 70GGC, MD98-2162, MD98-2165",Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-9.6500,-3.5660,115.200,119.383,-649,-482,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
8,51911,SMAK_d18Osw_comp,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[d18Osw, age_calBP, num_meas, d18Osw_err, d18O...",NOAA Template File,60179,"Cores 13GGC, 70GGC, MD98-2162, MD98-2165",Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-9.6500,-3.5660,115.200,119.383,-649,-482,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...
9,51912,WPAC_d18Osw_comp,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[age_calBP, num_meas, d18Osw_err, d18Osw_up, d...",NOAA Template File,60180,"Cores MD97-2141, MD98-2181, MD98-2176",Ocean>Pacific Ocean>Western Pacific Ocean,Feature,POLYGON,-5.0000,8.7830,121.283,133.433,-649,-482,10420,Makassar Strait 14KYr Foraminiferal Mg/Ca SST ...


## Summary

This notebook walks through way PyleoTUPS handles the NOAA response internally and the functionalities supporting access to the data and metadata using examples based on giving a NOAA ID. We understand that most users will want to access the NOAA querying capabilities, which is the subject of the [third tutorial](02_c_NOAASearch.ipynb) in this section. 